# Delayed Reward Optimization in Deep RL

## Overview

This notebook provides a deep technical exploration of the **delayed reward problem** in reinforcement learning, and how the Node2Vec + InferNet architecture solves it.

---

## The Delayed Reward Problem

In standard RL, the agent learns by maximizing cumulative reward:
$$G_t = \sum_{k=0}^{\infty} \gamma^k r_{t+k}$$

When rewards are **delayed** (sparse), credit assignment becomes extremely difficult:
- The agent receives reward at step $T$ but the key action was taken at step $T - \Delta$
- TD learning must propagate the credit backwards $\Delta$ steps
- With $\gamma < 1$, each step of propagation reduces the credit signal

### Reward Sparsity in Our Environment

Our 8×8 grid has only 3 coin nodes out of 64 total:
$$\text{Sparsity} = \frac{3}{64} \approx 4.7\%$$

This means a random policy discovers coins with probability ~4.7% per step.

## System Architecture

```mermaid
graph TD
    A[Graph Navigation Environment] --> B[Random Walk Sampler]
    B --> C[Node2Vec Model]
    C --> D[Node Embeddings]
    D --> E[InferNet]
    E --> F[Predicted Rewards]
    A --> G[Episode Trajectories]
    G --> H[Q-Table Update]
    F --> H
    H --> I[Greedy Policy]
    I --> A
```

### Three-Component Solution

1. **Node2Vec**: Learns structural representations of graph nodes
2. **InferNet**: Predicts rewards from structural representations  
3. **Q-Learning**: Propagates value through credit assignment

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import torch

from src.envs.graph_nav_env import GraphNavEnv, EnvConfig
from src.agents.node2vec_rl import Node2VecRLAgent, AgentConfig
from src.training.trainer import Trainer, TrainerConfig

print('Environment ready')

## Environment Setup & Visualization

In [ ]:
# Create the 8x8 graph navigation environment
env_config = EnvConfig(
    grid_size=8,
    coin_nodes={10, 30, 50},
    coin_reward=1.0,
    max_steps=128
)
env = GraphNavEnv(config=env_config)

print(f'Grid: {env.G}×{env.G} = {env.num_nodes} nodes')
print(f'Coin nodes: {env_config.coin_nodes}')
print(f'Sparsity: {len(env_config.coin_nodes)/env.num_nodes:.1%}')
print(f'Actions: {env.num_actions} (Up/Right/Down/Left)')

In [ ]:
# Visualize reward landscape
fig, ax = plt.subplots(figsize=(8, 8))
grid = np.zeros((8, 8))
for node in env_config.coin_nodes:
    r, c = node // 8, node % 8
    grid[r, c] = 1.0

im = ax.imshow(grid, cmap='RdYlGn', vmin=0, vmax=1, aspect='equal')
ax.set_title('Reward Landscape (green = coin)', fontsize=14, fontweight='bold')
ax.set_xlabel('Column'); ax.set_ylabel('Row')
for i in range(8):
    for j in range(8):
        node = i * 8 + j
        ax.text(j, i, str(node), ha='center', va='center', fontsize=7, color='black')
plt.colorbar(im, ax=ax, label='Reward')
plt.tight_layout()
plt.show()

## Training the Node2Vec RL Agent

In [ ]:
# Configure and train agent
agent_config = AgentConfig(
    embed_dim=512,
    num_iter=100,
    lr_node2vec=0.1,
    lr_infernet=0.01,
    gamma=0.99,
)
agent = Node2VecRLAgent(env, agent_config)

trainer_config = TrainerConfig(
    experiment_name='delayed_reward_demo',
    num_iterations=100,
    seed=42,
    checkpoint_dir='/tmp/checkpoints',
    output_dir='/tmp/outputs',
)
trainer = Trainer(agent, env, trainer_config)
results = trainer.train()

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(agent.losses_node2vec, color='steelblue', linewidth=1.5)
axes[0].set_title('Node2Vec Loss', fontweight='bold')
axes[0].set_xlabel('Iteration'); axes[0].set_ylabel('Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(agent.losses_infernet, color='darkorange', linewidth=1.5)
axes[1].set_title('InferNet Loss', fontweight='bold')
axes[1].set_xlabel('Iteration'); axes[1].set_ylabel('Loss')
axes[1].grid(True, alpha=0.3)

axes[2].plot(agent.episode_returns, color='green', linewidth=1.5)
axes[2].set_title('Mean Episode Return', fontweight='bold')
axes[2].set_xlabel('Iteration'); axes[2].set_ylabel('Return')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Policy Visualization

After training, we visualize the learned policy as directional arrows on the grid.

In [ ]:
print(env.render_policy(agent.policy))

## Credit Assignment Analysis

We can measure the credit assignment quality by computing the Q-value gradient—how quickly value propagates away from coin nodes.

In [ ]:
# Visualize Q-value landscape
max_q = agent.Q.max(dim=1)[0].detach().numpy().reshape(8, 8)

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(max_q, cmap='hot', aspect='equal')
ax.set_title('max Q(s,a) — Learned Value Landscape', fontsize=14, fontweight='bold')
plt.colorbar(im, ax=ax, label='Max Q-value')

# Mark coin nodes
for node in env_config.coin_nodes:
    r, c = node // 8, node % 8
    ax.add_patch(plt.Circle((c, r), 0.4, color='cyan', alpha=0.7))
    ax.text(c, r, '$', ha='center', va='center', fontsize=16, color='black', fontweight='bold')

plt.tight_layout()
plt.show()